<a href="https://colab.research.google.com/github/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_2/lessons/lesson_28_concurrency_intro/note_lesson_28_concurrency.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚡ Урок 28 — Потоки, процеси, asyncio: ранковий звіт диспетчерської

Диспетчерська «Смачно + Таксі» щоранку опитує 5 ресторанів (кожна відповідь — 0,5 с очікування мережі) і рахує важку статистику по 4 районах. Сьогодні пришвидшуємо обидві задачі — і вчимося обирати інструмент.

| Крок | Що робимо |
|---|---|
| 0 | задачі, що **чекають**, і задачі, що **рахують**; конкурентність і паралелізм |
| 1 | потоки: `ThreadPoolExecutor`, `max_workers` (вправа 1) |
| 2 | стан гонитви й `Lock`, «перевірив — і запізно» (вправи 2–4) |
| 3 | GIL і процеси: `ProcessPoolExecutor`, ціна процесів, `pickle` (вправа 5) |
| 4 | `asyncio`: `async def`, `await`, `gather`, тайм-аут (вправи 6–7) |
| 5 | знайди помилку (вправа 8) |
| 6 | блокуючий код в `asyncio`: `to_thread` (вправа 9) |
| 7 | обмежити одночасність: `Semaphore` (вправа 10) і що обрати |

**Як працювати:** виконуй клітинки **зверху вниз**. Перед **🔮 Прогнозом** спершу відповідай сам. Час у тебе буде трохи іншим, ніж у тексті, — важливе співвідношення. Теорія й покрокові схеми — у книзі: [Урок 28](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m2/lesson_28/).

---
## 0. Чекати чи рахувати

Функції для **процесів** мають жити у **файлі**: на Windows і macOS процес-працівник імпортує їх заново, а клітинку ноутбука імпортувати не можна. Тому запишемо їх у модуль `district_stats.py` — клітинка з `%%writefile` створює файл поруч з ноутбуком (і в Jupyter, і в Colab).

In [1]:
%%writefile district_stats.py
"""Функції, які запускаємо в процесах: мають бути у файлі, щоб процеси могли їх імпортувати."""

orders = []


def count_primes(limit):
    """Кількість простих чисел, менших за limit. Навмисно повільно: перебір дільників."""
    count = 0
    for n in range(2, limit):
        for d in range(2, int(n ** 0.5) + 1):
            if n % d == 0:
                break
        else:
            count += 1
    return count


def add_order(name):
    orders.append(name)      # додає у СВОЮ копію списку
    return len(orders)

Overwriting district_stats.py


In [ ]:
import asyncio
import threading
import time
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor

import district_stats

RESTAURANTS = ["Борщ і Ко", "Піца Поділ", "Суші Оболонь", "Вареники 24/7", "Шаурма Центр"]
DISTRICTS = [150_000] * 4


def check_restaurant(name):
    time.sleep(0.5)            # чекаємо відповідь мережі
    return f"{name}: відкрито"


def timed(label, work):
    start = time.perf_counter()
    result = work()
    print(f"{label}: {time.perf_counter() - start:.1f} с")
    return result


timed("5 ресторанів послідовно", lambda: [check_restaurant(n) for n in RESTAURANTS])
timed("4 райони послідовно", lambda: [district_stats.count_primes(n) for n in DISTRICTS])

Ресторани — задача, що **чекає** (IO-bound): процесор простоює, поки йде відповідь. Райони — задача, що **рахує** (CPU-bound): процесор зайнятий на 100 %.

Два слова, які далі знадобляться:

- **Конкурентність** — кілька задач просуваються за той самий проміжок часу, але не обов'язково в ту саму мить. П'ять ресторанів, які ми чекаємо разом, — конкурентність.
- **Паралелізм** — робота виконується буквально одночасно, наприклад на кількох ядрах процесора.

Інструмент обираємо за типом задачі — і завжди **міряємо**, а не вгадуємо.

### Вправа 0. Чекає чи рахує?

Для кожної задачі запиши `"IO"` (чекає) або `"CPU"` (рахує).

In [ ]:
kinds = {
    # YOUR CODE HERE
    # BEGIN SOLUTION
    "завантажити 100 сторінок меню з сайтів ресторанів": "IO",
    "стиснути 500 фото страв": "CPU",
    "прочитати 20 файлів звітів з диска": "IO",
    "порахувати найкоротші маршрути для 10 000 поїздок": "CPU",
    "надіслати 1000 SMS через API": "IO",
    # END SOLUTION
}

assert kinds["завантажити 100 сторінок меню з сайтів ресторанів"] == "IO"
assert kinds["стиснути 500 фото страв"] == "CPU"
assert kinds["прочитати 20 файлів звітів з диска"] == "IO"
assert kinds["порахувати найкоротші маршрути для 10 000 поїздок"] == "CPU"
assert kinds["надіслати 1000 SMS через API"] == "IO"
print("✅ Вправа 0 пройдена")

---
## 1. Потоки: чекати одночасно

**🔮 Прогноз:** скільки займе опитування 5 ресторанів через `ThreadPoolExecutor`? У якому порядку будуть результати?

<details>
<summary>Відповідь</summary>

Близько 0,5 с: п'ять очікувань ідуть одночасно. `pool.map` повертає результати **в порядку вхідного списку**, хоч би в якому порядку закінчились потоки.

</details>

In [ ]:
with ThreadPoolExecutor() as pool:
    results = timed("5 ресторанів у потоках", lambda: list(pool.map(check_restaurant, RESTAURANTS)))
print(results)

### Вправа 1. `check_all(names)`

Поверни список статусів для всіх ресторанів, опитуючи їх у потоках. Порядок — як у `names`.

In [ ]:
def check_all(names):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    with ThreadPoolExecutor() as pool:
        return list(pool.map(check_restaurant, names))
    # END SOLUTION


start = time.perf_counter()
statuses = check_all(RESTAURANTS)
elapsed = time.perf_counter() - start
print(statuses[0], f"| {elapsed:.1f} с")
assert statuses == [f"{n}: відкрито" for n in RESTAURANTS]
assert elapsed < 1.5, "Схоже, ресторани опитуються по черзі"
print("✅ Вправа 1 пройдена")

**Потік** — окрема лінія виконання всередині однієї програми (процесу). Потоки допомагають ресторанам, бо під час `time.sleep` чи мережевого запиту потік **відпускає GIL** (розділ 3) — і поки він чекає, працюють інші.

`with` сам закриває пул: дочекається всіх потоків і прибере їх.

**🔮 Прогноз:** скільки займе опитування, якщо обмежити пул двома потоками — `max_workers=2`?

<details>
<summary>Відповідь</summary>

Близько 1,5 с. Одночасно працюють лише два потоки: ресторани йдуть «хвилями» 2 + 2 + 1, кожна по 0,5 с. Без `max_workers` пул бере `min(32, кількість ядер + 4)` потоків — тому вище всі п'ять ішли разом. Для порівнянь задавай `max_workers` явно, щоб результат не залежав від машини.

</details>

In [ ]:
with ThreadPoolExecutor(max_workers=2) as pool:
    timed("5 ресторанів, 2 потоки", lambda: list(pool.map(check_restaurant, RESTAURANTS)))

Пул — не єдиний спосіб. Можна створити потік вручну через `threading.Thread`, але він **викидає** те, що повертає функція: результат доведеться складати у спільну змінну самому.

In [ ]:
statuses_by_hand = {}


def check_into_dict(name):
    statuses_by_hand[name] = check_restaurant(name)   # return тут нікому не потрібен


thread = threading.Thread(target=check_into_dict, args=("Борщ і Ко",))
thread.start()
thread.join()                                          # чекаємо, поки потік закінчить
print(statuses_by_hand)

Спільна змінна між потоками — це саме те місце, де починаються проблеми. Про них — наступний розділ. `pool.map` повертає результати сам, тому для більшості задач він простіший.

---
## 2. Стан гонитви

Чотири каси одночасно зараховують по 1000 оплат по 1 грн. «Прочитати» і «записати» — два кроки, а між ними інший потік встигає своє. `time.sleep(0)` лише віддає чергу іншому потоку, щоб проблема проявилася наочно.

**🔮 Прогноз:** скільки буде на рахунку після роботи чотирьох кас?

<details>
<summary>Відповідь</summary>

Значно менше за 4000 — щоразу інше число (у наших 30 запусках — від 1000 до 1336). Каси читають старе значення й записують поверх чужих оплат.

</details>

In [ ]:
balance = 0


def pay(times):
    global balance
    for _ in range(times):
        current = balance        # 1. прочитати
        time.sleep(0)            # каса на мить відволіклася
        balance = current + 1    # 2. записати


threads = [threading.Thread(target=pay, args=(1000,)) for _ in range(4)]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()
print("Очікували 4000, на рахунку:", balance)

### Вправа 2. Виправ через `Lock`

Допиши `pay_safe`: блок «прочитати — записати» має виконуватись під `with lock:`.

In [ ]:
balance = 0
lock = threading.Lock()


def pay_safe(times):
    global balance
    for _ in range(times):
        # YOUR CODE HERE
        # BEGIN SOLUTION
        with lock:
            current = balance
            time.sleep(0)
            balance = current + 1
        # END SOLUTION


threads = [threading.Thread(target=pay_safe, args=(1000,)) for _ in range(4)]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()
print("На рахунку:", balance)
assert balance == 4000
print("✅ Вправа 2 пройдена")

### Вправа 3. Без спільних змінних

Кращий шлях — не ділити змінну взагалі. Нехай кожна каса **повертає**, скільки оплат прийняла, а головна програма підсумовує результати `pool.map`. Жодного `global`, жодного `Lock`.

In [ ]:
def count_payments(times):
    """Каса приймає times оплат по 1 грн і повертає свою суму."""
    # YOUR CODE HERE
    # BEGIN SOLUTION
    total = 0
    for _ in range(times):
        time.sleep(0)
        total += 1
    return total
    # END SOLUTION


def total_payments(cashboxes):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    with ThreadPoolExecutor() as pool:
        return sum(pool.map(count_payments, cashboxes))
    # END SOLUTION


print(total_payments([1000, 1000, 1000, 1000]))
assert total_payments([1000, 1000, 1000, 1000]) == 4000
assert total_payments([5, 7]) == 12
assert total_payments([]) == 0
print("✅ Вправа 3 пройдена")

### Перевірив — і вже запізно

Диспетчерська імпортує меню ресторанів і веде список `imported` — хто вже імпортований. Деякі назви приходять у завдання кілька разів. Код перевіряє «чи вже є?» і лише тоді дописує. Між перевіркою й записом — збереження меню в базу (0,01 с).

**🔮 Прогноз:** 8 завдань, але лише 2 різні ресторани. Скільки записів буде в `imported`?

<details>
<summary>Відповідь</summary>

8 — кожна назва чотири рази. Потоки одночасно прокидаються після мережі, усі четверо бачать «ще немає» і всі четверо дописують. Це **check-then-act** («перевір, потім дій») — класичний стан гонитви.

**GIL тут не рятує.** Він не робить операцію з кількох кроків атомарною: між `if` і `append` інший потік встигає своє.

</details>

In [ ]:
TASKS = ["Борщ і Ко", "Піца Поділ"] * 4      # 8 завдань, 2 ресторани


def save_menu(name):
    time.sleep(0.01)                         # пишемо меню в базу


imported = []


def import_menu(name):
    check_restaurant(name)                   # завантаження: 0,5 с мережі
    if name not in imported:                 # 1. перевірити
        save_menu(name)
        imported.append(name)                # 2. дописати


with ThreadPoolExecutor(max_workers=8) as pool:
    list(pool.map(import_menu, TASKS))
print(len(imported), "записів:", imported)

### Вправа 4. Перевірка й запис — під одним замком

Допиши `import_menu_safe` повністю. Правила:

- `if name not in imported` і `imported.append(name)` — разом під `with imported_lock:`; між ними — `save_menu(name)`;
- повільне завантаження `check_restaurant(name)` — **поза** замком. Інакше вісім завантажень підуть по черзі (4 с), і потоки втратять сенс.

In [ ]:
imported = []
imported_lock = threading.Lock()


def import_menu_safe(name):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    check_restaurant(name)                   # повільне — поза замком
    with imported_lock:
        if name not in imported:
            save_menu(name)
            imported.append(name)
    # END SOLUTION


start = time.perf_counter()
with ThreadPoolExecutor(max_workers=8) as pool:
    list(pool.map(import_menu_safe, TASKS))
elapsed = time.perf_counter() - start
print(imported, f"| {elapsed:.1f} с")
assert sorted(imported) == ["Борщ і Ко", "Піца Поділ"], "У списку дублікати: перевірка й запис мають бути під одним замком"
assert elapsed < 1.5, "Завантаження опинилось під замком — винеси check_restaurant за with"
print("✅ Вправа 4 пройдена")

Замок прибрав дублікати **в списку**, але «Борщ і Ко» все одно **завантажили** чотири рази. Щоб не качати двічі, треба до завантаження позначати ресторан як «вже в роботі» і окремо вирішувати, що робити, якщо завантаження впало. Це вже інша, складніша задача.

---
## 3. GIL і процеси

**GIL** (Global Interpreter Lock) — замок у звичайному CPython: у кожен момент байт-код Python виконує **лише один потік**. Під час очікування (I/O) потік його відпускає — тому потоки й допомогли ресторанам. А під час обчислень на чистому Python потоки лише чергуються. Бібліотеки, написані на C (наприклад, NumPy), можуть відпускати GIL і під час обчислень — там картина інша.

З Python 3.14 є окрема, офіційно підтримувана збірка **без GIL** (free-threaded). Її видно в `python -VV` (рядок містить `free-threading build`). Цей урок — про звичайну збірку з GIL. Перевір свою:

In [ ]:
import sys

print(sys.version)
gil_check = getattr(sys, "_is_gil_enabled", None)      # з'явилась у Python 3.13
print("GIL увімкнено:", gil_check() if gil_check else True)

**🔮 Прогноз:** чи пришвидшать 4 потоки підрахунок `count_primes` для 4 районів? А 4 процеси?

<details>
<summary>Відповідь</summary>

Потоки — ні: через **GIL** байт-код у кожен момент виконує лише один потік, вони чергуються. Процеси — так: у кожного свій інтерпретатор і свій GIL, вони рахують на різних ядрах. На нашій 4-ядерній машині: послідовно ≈0,7 с, потоки ≈0,7–0,8 с, процеси ≈0,2 с.

</details>

In [ ]:
timed("послідовно", lambda: [district_stats.count_primes(n) for n in DISTRICTS])
with ThreadPoolExecutor(max_workers=4) as pool:
    timed("4 потоки  ", lambda: list(pool.map(district_stats.count_primes, DISTRICTS)))
with ProcessPoolExecutor(max_workers=4) as pool:
    timed("4 процеси ", lambda: list(pool.map(district_stats.count_primes, DISTRICTS)))

У ноутбуці немає `if __name__ == "__main__":` — він і так головна програма. У `.py`-файлі код, що створює процеси, **обов'язково** стоїть під цією перевіркою. Ось чому.

Python запускає процес-працівник одним із **способів старту**. Windows і macOS використовують `spawn`, Linux з Python 3.14 — `forkserver` (до 3.14 — `fork`). `spawn` і `forkserver` стартують чистий інтерпретатор, який **заново імпортує** твій головний файл, щоб знайти функцію. Якби пул створювався просто на рівні файлу, кожен імпорт створював би новий пул — і Python зупиняє це з `RuntimeError` про «bootstrapping phase». Код під `if __name__ == "__main__":` під час такого імпорту не виконується.


In [ ]:
import multiprocessing

print("Спосіб старту процесів тут:", multiprocessing.get_start_method())

Процеси не діляться пам'яттю — перевір:

In [ ]:
district_stats.orders.clear()
with ProcessPoolExecutor(max_workers=1) as pool:
    print(list(pool.map(district_stats.add_order, ["А", "Б", "В"])))
print("orders у ноутбуці:", district_stats.orders)

Дані між процесами ходять лише **аргументами й результатами**. Python пакує їх через `pickle` і пересилає. Звідси два наслідки.

**1. Не все можна передати.** Функція має бути **на верхньому рівні модуля**: lambda чи функцію, визначену всередині іншої функції, `pickle` не запакує. Відкритий файл чи з'єднання з базою теж не передаси — передавай шлях, а відкривай уже в працівнику.

In [ ]:
try:
    with ProcessPoolExecutor(max_workers=1) as pool:
        print(list(pool.map(lambda n: n * 2, [1, 2, 3])))
except Exception as error:
    print(type(error).__name__, "—", error)

**2. Процеси не безкоштовні.** Запустити працівника, запакувати й переслати дані — це час.

**🔮 Прогноз:** 4 крихітні задачі `count_primes(100)` — що швидше: послідовно чи в 4 процесах (разом зі стартом пулу)?

<details>
<summary>Відповідь</summary>

Послідовно — частки мілісекунди. Процеси — у сотні разів довше: майже весь час іде на старт працівників і пересилання даних. На Windows і macOS (`spawn`) різниця ще більша. Процеси окупаються, коли кожна задача рахує помітний час, а даних ходить мало. Тому — спершу виміряй.

</details>

In [ ]:
def in_processes(func, data):
    with ProcessPoolExecutor(max_workers=4) as pool:      # старт пулу теж рахуємо
        return list(pool.map(func, data))


SMALL = [100] * 4

start = time.perf_counter()
[district_stats.count_primes(n) for n in SMALL]
sequential = time.perf_counter() - start

start = time.perf_counter()
in_processes(district_stats.count_primes, SMALL)
processes = time.perf_counter() - start

print(f"малі задачі послідовно: {sequential:.4f} с")
print(f"малі задачі в процесах: {processes:.4f} с — у {processes / sequential:.0f} разів повільніше")

### Вправа 5. `district_report(limits)`

Поверни список результатів `district_stats.count_primes` для кожного ліміту, рахуючи в процесах. Порядок — як у `limits`.

In [ ]:
def district_report(limits):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    with ProcessPoolExecutor() as pool:
        return list(pool.map(district_stats.count_primes, limits))
    # END SOLUTION


report = district_report([10, 100, 1000, 150_000])
print(report)
assert report == [4, 25, 168, 13848]
print("✅ Вправа 5 пройдена")

`async def` створює **корутину**: виклик не виконує тіло, а повертає об'єкт — як генераторна функція (урок 11). Запускає корутини **цикл подій** (event loop) — диспетчер, який виконує задачі по черзі в одному потоці й перемикається, коли задача чекає. Місце, де можна стати на паузу, позначає `await`. Пауза буде, лише якщо операції справді треба чекати: `await` на готовому результаті просто йде далі.

У Jupyter і Colab цикл подій уже працює, тому **замість `asyncio.run(main())` пиши `await main()`** прямо в клітинці. У `.py`-файлі — навпаки, `asyncio.run(main())`: він створює цикл, виконує корутину й закриває цикл.


In [ ]:
async def check_restaurant_async(name):
    await asyncio.sleep(0.5)          # чекаємо, але не блокуємо інших
    return f"{name}: відкрито"


coro = check_restaurant_async("Борщ і Ко")
print(type(coro).__name__)
print(await coro)                     # у файлі: asyncio.run(...)

**🔮 Прогноз:** що надрукує `kitchen()` і в якому порядку?

```python
async def cook(name, seconds):
    print(f"{name}: почав")
    await asyncio.sleep(seconds)
    print(f"{name}: закінчив")


async def kitchen():
    await asyncio.gather(cook("борщ", 0.2), cook("салат", 0.1))
```

<details>
<summary>Відповідь</summary>

`борщ: почав`, `салат: почав`, `салат: закінчив`, `борщ: закінчив`. Борщ доходить до `await` і стає на паузу — цикл подій запускає салат. Салат чекає менше, тому закінчує першим.

</details>

In [ ]:
async def cook(name, seconds):
    print(f"{name}: почав")
    await asyncio.sleep(seconds)
    print(f"{name}: закінчив")


async def kitchen():
    await asyncio.gather(cook("борщ", 0.2), cook("салат", 0.1))


await kitchen()

**🔮 Прогноз:** колега написав опитування з `await`, але без `gather`. Скільки воно займе?

```python
async def check_one_by_one(names):
    results = []
    for name in names:
        results.append(await check_restaurant_async(name))
    return results
```

<details>
<summary>Відповідь</summary>

2,5 с — як без `asyncio`. `await` чекає, поки **ця** корутина закінчиться, і лише потім цикл `for` бере наступний ресторан. Одночасно корутини йдуть, лише коли їх запущено разом: `gather` (або задачі `asyncio.create_task`).

</details>

In [ ]:
async def check_one_by_one(names):
    results = []
    for name in names:
        results.append(await check_restaurant_async(name))
    return results


start = time.perf_counter()
await check_one_by_one(RESTAURANTS)
print(f"await у циклі: {time.perf_counter() - start:.1f} с")

А якщо корутину створили й **забули** — без `await` і без `gather`? Її тіло так і не виконається, а Python, прибираючи об'єкт, попередить `RuntimeWarning`. Зловимо це попередження:

In [ ]:
import gc
import warnings

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    check_restaurant_async("Піца Поділ")      # створили й забули: тіло не виконалось
    gc.collect()

for warning in caught:
    print(warning.category.__name__, "—", warning.message)

### Вправа 6. `check_all_async(names)`

Корутина: опитати всі ресторани одночасно через `asyncio.gather` і повернути список статусів у порядку `names`.

In [ ]:
async def check_all_async(names):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    return await asyncio.gather(*(check_restaurant_async(n) for n in names))
    # END SOLUTION


start = time.perf_counter()
statuses = await check_all_async(RESTAURANTS)
elapsed = time.perf_counter() - start
print(statuses[-1], f"| {elapsed:.1f} с")
assert list(statuses) == [f"{n}: відкрито" for n in RESTAURANTS]
assert elapsed < 1.5
print("✅ Вправа 6 пройдена")

### Вправа 7. Тайм-аут

Ресторан, що не відповів за `timeout` секунд, отримує статус `"<назва>: не відповідає"`. Використай `asyncio.wait_for(корутина, timeout=...)` і `except asyncio.TimeoutError` (на Python 3.10 це не той самий клас, що вбудований `TimeoutError`; з 3.11 — той самий).

In [ ]:
async def check_with_delay(name, delay):
    await asyncio.sleep(delay)
    return f"{name}: відкрито"


async def safe_check(name, delay, timeout=1.0):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    try:
        return await asyncio.wait_for(check_with_delay(name, delay), timeout=timeout)
    except asyncio.TimeoutError:
        return f"{name}: не відповідає"
    # END SOLUTION


delays = {"Борщ і Ко": 3.0, "Піца Поділ": 0.5, "Суші Оболонь": 0.5, "Вареники 24/7": 0.5, "Шаурма Центр": 0.5}
start = time.perf_counter()
lines = await asyncio.gather(*(safe_check(n, d) for n, d in delays.items()))
elapsed = time.perf_counter() - start
for line in lines:
    print(line)
print(f"{elapsed:.1f} с")
assert lines[0] == "Борщ і Ко: не відповідає"
assert all(line.endswith("відкрито") for line in lines[1:])
assert elapsed < 2.0
print("✅ Вправа 7 пройдена")

---
## 5. Знайди помилку

### Вправа 8

Колега переписав опитування на `asyncio`, але воно й далі триває 2,5 с. Ось його `check_slow`:

```python
async def check_slow(name):
    time.sleep(0.5)
    return f"{name}: відкрито"
```

Знайди причину й напиши виправлене тіло `check_slow` у клітинці нижче, не змінюючи `report_slow`.

In [ ]:
async def check_slow(name):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    await asyncio.sleep(0.5)
    # END SOLUTION
    return f"{name}: відкрито"


async def report_slow():
    return await asyncio.gather(*(check_slow(n) for n in RESTAURANTS))


start = time.perf_counter()
await report_slow()
elapsed = time.perf_counter() - start
print(f"{elapsed:.1f} с")
assert elapsed < 1.5, "Корутини досі виконуються по черзі — шукай блокуючий виклик"
print("✅ Вправа 8 пройдена")

---
## 6. Блокуючий код, який не можна переписати

Вправа 8 виправлялась легко: `time.sleep` → `await asyncio.sleep`. Але буває, що блокуючу функцію переписати не можна: стара бібліотека, `requests.get`, синхронний драйвер бази. Такий виклик в `async def` зупинить увесь цикл подій.

Вихід — `await asyncio.to_thread(функція, аргументи...)`: функція виконується в окремому потоці, а корутина чекає її, не блокуючи цикл.

### Вправа 9. `check_all_blocking(names)`

Опитай ресторани **звичайною** `check_restaurant` (вона блокує, її не чіпаємо), але одночасно: `gather` + `asyncio.to_thread`.

In [ ]:
async def check_all_blocking(names):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    return await asyncio.gather(*(asyncio.to_thread(check_restaurant, n) for n in names))
    # END SOLUTION


start = time.perf_counter()
statuses = await check_all_blocking(RESTAURANTS)
elapsed = time.perf_counter() - start
print(statuses[0], f"| {elapsed:.1f} с")
assert list(statuses) == [f"{n}: відкрито" for n in RESTAURANTS]
assert elapsed < 1.5, "Виклики досі йдуть по черзі — загорни check_restaurant у asyncio.to_thread"
print("✅ Вправа 9 пройдена")

`to_thread` бере потоки зі стандартного пулу циклу подій (`min(32, ядра + 4)`), тож на тисячах викликів час упреться в цей розмір. А для **обчислень** на чистому Python `to_thread` не допоможе — GIL той самий. Їх відправляють у пул процесів.

---
## 7. Обмежити одночасність: `asyncio.Semaphore`

`gather` запускає **все одразу**: 5 ресторанів — 5 запитів, 5000 — 5000. Сервер може такого не витримати або заблокувати тебе. У пулі потоків межу задає `max_workers`, в `asyncio` — **семафор**: `asyncio.Semaphore(2)` пускає в захищений блок `async with semaphore:` не більше двох корутин одночасно, решта чекають біля входу.

**🔮 Прогноз:** 5 ресторанів по 0,5 с, межа — 2. Скільки часу?

<details>
<summary>Відповідь</summary>

Близько 1,5 с: три «хвилі» — 2 + 2 + 1.

</details>

### Вправа 10. `check_limited(names, limit)`

Допиши `one(name)`: виклик `check_counted(name)` має бути всередині `async with semaphore:`. `check_counted` сама рахує, скільки запитів «в польоті» одночасно.

In [ ]:
in_flight = 0
max_in_flight = 0


async def check_counted(name):
    global in_flight, max_in_flight
    in_flight += 1
    max_in_flight = max(max_in_flight, in_flight)
    await asyncio.sleep(0.5)
    in_flight -= 1
    return f"{name}: відкрито"


async def check_limited(names, limit):
    semaphore = asyncio.Semaphore(limit)

    async def one(name):
        # YOUR CODE HERE
        # BEGIN SOLUTION
        async with semaphore:
            return await check_counted(name)
        # END SOLUTION

    return await asyncio.gather(*(one(n) for n in names))


start = time.perf_counter()
statuses = await check_limited(RESTAURANTS, limit=2)
elapsed = time.perf_counter() - start
print(f"одночасно щонайбільше: {max_in_flight} | {elapsed:.1f} с")
assert list(statuses) == [f"{n}: відкрито" for n in RESTAURANTS]
assert max_in_flight == 2, "Одночасно мають працювати рівно 2 запити"
assert 1.3 < elapsed < 2.2
print("✅ Вправа 10 пройдена")

Семафор обмежує, скільки запитів **у польоті одночасно**, а не скільки їх **за секунду**. Ліміти API («не більше 10 запитів за секунду») — окрема річ. Якщо сервер відповів `429 Too Many Requests`, він часто додає заголовок `Retry-After` — скільки секунд почекати. Тоді чекають і повторюють, щоразу довше (backoff). Детальніше — у книзі й в уроці 32.

---
## Що обрати

| Робота | З чого почати |
|---|---|
| Чекає, десятки задач, звичайний (синхронний) код | `ThreadPoolExecutor` |
| Чекає, багато одночасних з'єднань, асинхронні бібліотеки | `asyncio` |
| Рахує на чистому Python, GIL увімкнено | `ProcessPoolExecutor` — після виміру накладних витрат |
| Чекає **і** рахує | завантажити потоками чи `asyncio`, потім рахувати в пулі процесів |

Обирай найпростіший інструмент, що підходить, **обмежуй** одночасність і **міряй**.

---
## Самоперевірка

1. Чим задача, що чекає, відрізняється від задачі, що рахує? Що прискорює кожну?
2. Чому потоки не прискорили `count_primes`?
3. Як уникнути стану гонитви без `Lock`?
4. Чому `district_stats.orders` у ноутбуці лишився порожнім?
5. Чим `asyncio.run(main())` у файлі відрізняється від `await main()` у ноутбуці?
6. Чому перевірку `if name not in imported` і `append` треба тримати під одним замком, а завантаження — поза ним?
7. Коли пул процесів повільніший за звичайний цикл? Чому lambda не можна передати в `ProcessPoolExecutor`?
8. Чому `await` у циклі `for` не пришвидшує опитування?
9. Як викликати блокуючий `requests.get` з асинхронного коду, не зупиняючи цикл подій?
10. Що обмежує `asyncio.Semaphore(3)`, а чого — ні?

<details>
<summary>Відповіді</summary>

1. Задача, що чекає, здебільшого простоює (мережа, диск) — допомагають потоки або `asyncio`. Задача, що рахує, займає процесор — допомагають процеси.
2. Через GIL: у кожен момент байт-код виконує лише один потік програми.
3. Не ділити змінних: кожен потік повертає результат, головна програма їх збирає.
4. Процес-працівник має власну копію пам'яті; дані повертаються лише результатами функцій.
5. `asyncio.run` створює цикл подій; у ноутбуці цикл уже працює, тому корутину просто чекають через `await`.
6. Інакше два потоки обидва побачать «ще немає» і обидва допишуть (check-then-act). Завантаження під замком пішли б по черзі — потоки втратили б сенс.
7. Коли задачі дрібні: старт працівників і пересилання даних дорожчі за саму роботу. Аргументи й функції пакуються через `pickle`, а lambda він не пакує — потрібна функція верхнього рівня модуля.
8. `await` чекає, поки закінчиться ця корутина, і лише потім цикл бере наступну. Одночасно корутини йдуть через `gather` чи `create_task`.
9. `await asyncio.to_thread(requests.get, url)` — виклик іде в окремому потоці.
10. Скільки корутин одночасно в захищеному блоці. Не обмежує кількість запитів за секунду — це окремо: паузи, `Retry-After`, backoff.

</details>

## Далі

- Теорія, покрокові схеми гонитви й циклу подій, вибір інструменту: [Урок 28](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m2/lesson_28/).
- Документація: [`concurrent.futures`](https://docs.python.org/3/library/concurrent.futures.html), [`threading`](https://docs.python.org/3/library/threading.html), [`multiprocessing`](https://docs.python.org/3/library/multiprocessing.html), [`asyncio` — Coroutines and Tasks](https://docs.python.org/3/library/asyncio-task.html) (розділи «Coroutines», «Awaitables», «Running Tasks Concurrently», «Running in Threads»).
- Почитати після уроку: [Real Python — Speed Up Your Python Program With Concurrency](https://realpython.com/python-concurrency/) (≈40 хв), [SuperFastPython — як обрати API для конкурентності](https://superfastpython.com/python-concurrency-choose-api/).
- Без GIL: [Python support for free threading](https://docs.python.org/3/howto/free-threading-python.html), [PEP 779](https://peps.python.org/pep-0779/) — чому з 3.14 збірка без GIL офіційно підтримується.
- **Поглиблений рівень — бонусні уроки модуля 4** (`module_4/bonus/`). Там ці самі теми розписано значно детальніше, на реальних проєктах: [потоки й конкурентність](https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/bonus/lesson_threading_concurrency/note_lesson_32_threading_concurrency.ipynb) і [багатопотоковий скрейпер](https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/bonus/lesson_threading_concurrency/note_lesson_32_multithreaded_scraper.ipynb); [multiprocessing](https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/bonus/lesson_multiprocessing/note_lesson_33_multiprocessing.ipynb) і [лабораторна: паралельний конвеєр обробки даних](https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/bonus/lesson_multiprocessing/lab_lesson_33_flood_pipeline.ipynb); [asyncio](https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/bonus/lesson_asyncio/lesson_34_asyncio.ipynb) з асинхронним FastAPI-демо й новинним дашбордом. Перед ними корисні [мережа: сокети й TCP](https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/bonus/lesson_networking_base/note_lesson_30_networking_base.ipynb) і [HTTP-запити](https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/bonus/lesson_http_requests/note_lesson_31_http_requests.ipynb).
- **Урок 29** — практикум П6: стек, черга, купа, префіксне дерево й LRU-кеш.